In [1]:
from robustraster import dask_cluster_manager

json_key = r"R:\SCRATCH\adrianom\credentials\earthengineapi\robust-raster-cecdcc4b5fba.json"
dask_cluster = dask_cluster_manager.DaskClusterManager()
dask_cluster.create_cluster(mode="full", threads_per_worker=1, n_workers=19, memory_limit="10GB")

Dask dashboard is available at: http://127.0.0.1:8787/status


In [2]:
# 2. Authenticate Google Earth Engine on all Dask workers.
from robustraster import dask_plugins

ee_plugin = dask_plugins.EEPlugin(json_key)
dask_client = dask_cluster.get_dask_client
dask_client.register_plugin(ee_plugin)

{'tcp://127.0.0.1:51992': {'status': 'OK'},
 'tcp://127.0.0.1:51993': {'status': 'OK'},
 'tcp://127.0.0.1:51994': {'status': 'OK'},
 'tcp://127.0.0.1:52001': {'status': 'OK'},
 'tcp://127.0.0.1:52002': {'status': 'OK'},
 'tcp://127.0.0.1:52003': {'status': 'OK'},
 'tcp://127.0.0.1:52004': {'status': 'OK'},
 'tcp://127.0.0.1:52005': {'status': 'OK'},
 'tcp://127.0.0.1:52006': {'status': 'OK'},
 'tcp://127.0.0.1:52007': {'status': 'OK'},
 'tcp://127.0.0.1:52008': {'status': 'OK'},
 'tcp://127.0.0.1:52025': {'status': 'OK'},
 'tcp://127.0.0.1:52026': {'status': 'OK'},
 'tcp://127.0.0.1:52027': {'status': 'OK'},
 'tcp://127.0.0.1:52034': {'status': 'OK'},
 'tcp://127.0.0.1:52037': {'status': 'OK'},
 'tcp://127.0.0.1:52038': {'status': 'OK'},
 'tcp://127.0.0.1:52043': {'status': 'OK'},
 'tcp://127.0.0.1:52046': {'status': 'OK'}}

In [3]:
# 3. Obtain the header information for the Earth Engine query and store it in an xarray object.
#    This code does not do a full query for the data (yet)! 

#    In this example, we are just querying some data from Landsat 8 imagery 
#    over a small watershed for demo purposes.

from robustraster import dataset_manager
import ee
import json
import xarray as xr
from dask.distributed import performance_report
import dask

# Although we authenticated Google Earth Engine on our Dask workers, we also need to authenticate
# Google Earth Engine on our local machine!
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)
ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')


WSDemo = ee.FeatureCollection("projects/robust-raster/assets/boundaries/WSDemoSHP_Albers")


#ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2014-01-01', '2014-12-31')
ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2020-05-01', '2020-08-31').select(['SR_B4', 'SR_B5']).sort("system:time_start")
xarray_data = xr.open_dataset(ic,
            engine='ee', 
            crs="EPSG:3310", 
            scale=30,
            geometry=WSDemo.geometry(),
            chunks={"time": 48, "X": 512, "Y": 256})

r:\Users\adrianom.UNR\.conda\envs\robustrastertest\lib\site-packages\xarray\core\dataset.py:282: UserWarning: The specified chunks separate the stored chunks along dimension "Y" starting at index 256. This could degrade performance. Instead, consider rechunking after loading.
  warnings.warn(


In [7]:
print(xarray_data)

<xarray.Dataset> Size: 124GB
Dimensions:  (time: 61618, X: 459, Y: 546)
Coordinates:
  * time     (time) datetime64[ns] 493kB 2020-05-01T00:54:10.847000 ... 2020-...
  * X        (X) float64 4kB -7.668e+04 -7.665e+04 ... -6.297e+04 -6.294e+04
  * Y        (Y) float64 4kB 1.886e+05 1.886e+05 ... 2.049e+05 2.05e+05
Data variables:
    SR_B4    (time, X, Y) float32 62GB dask.array<chunksize=(48, 459, 256), meta=np.ndarray>
    SR_B5    (time, X, Y) float32 62GB dask.array<chunksize=(48, 459, 256), meta=np.ndarray>
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:                 0
    product_tags:           ['global', 'sr', 'reflectance', 'l8sr', 'lst', 'c...
    provider:               USGS
    ...                     ...
    visualization_1_name:   Near Infrared (543)
    visualization_

In [4]:
import dask

def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df

"""
@dask.delayed
def user_function_wrapper(ds, user_func=compute_ndvi, *args, **kwargs):
    df = ds.to_dataframe().reset_index()
    df = user_func(df, *args, **kwargs)
    df = df.set_index(list(ds.dims))
    ds_output = df.to_xarray()
    return ds_output

data = dask.compute(xarray_data)
"""

def user_function_wrapper(ds, user_func=compute_ndvi, *args, **kwargs):
    df = ds.to_dataframe().reset_index()
    df = user_func(df, *args, **kwargs)
    df = df.set_index(list(ds.dims))
    ds_output = df.to_xarray()
    return ds_output

test = xr.map_blocks(user_function_wrapper, xarray_data)

some = test.compute()


KeyboardInterrupt: 

In [5]:
import dask

@dask.delayed
def dataframe_conversion(ds):
    return ds.to_dataframe().reset_index()

@dask.delayed
def compute_ndvi(df):
    df["ndvi"] = (df["SR_B5"] - df["SR_B4"]) / (df["SR_B5"] + df["SR_B4"])
    return df

@dask.delayed
def dataframe_to_xarray(df, dims):
    df = df.set_index(dims)
    return df.to_xarray()

delayed_df = dataframe_conversion(xarray_data)
delayed_ndvi = compute_ndvi(delayed_df)
delayed_xr = dataframe_to_xarray(delayed_ndvi, list(xarray_data.dims))

# Compute everything at once
result = delayed_xr.compute()

KeyboardInterrupt: 